# Algorithms for massive data Project

## Data loading

In [1]:
import os
import zipfile
os.environ['KAGGLE_USERNAME'] = "xxxx"
os.environ['KAGGLE_KEY'] = "xxxx"
!kaggle datasets download -d Cornell-University/arxiv
with zipfile.ZipFile("arxiv.zip", "r") as zip_ref:
    zip_ref.extractall("arxiv_data")

"kaggle" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.


## Configuration

In [4]:
import json
DATA_FILE = "arxiv_data/arxiv-metadata-oai-snapshot.json"
MAX_RECORDS = 100000
MIN_PAPERS  = 2
DAMPING = 0.85
TOP_N = 20

## Parsing author names

In [5]:
def get_authors(record):
    parsed = record.get("authors_parsed") or []
    if parsed:
        names = []
        for entry in parsed:
            last = entry[0].strip() if len(entry) > 0 else ""
            first = entry[1].strip() if len(entry) > 1 else ""
            if last:
                names.append(f"{last}, {first}" if first else last)
        return names

    return [a.strip() for a in record.get("authors", "").split(",") if a.strip()]

## Authors count

In [6]:
paper_count = {}
n = 0

with zipfile.ZipFile("arxiv.zip", 'r') as z:
    with z.open("arxiv-metadata-oai-snapshot.json") as f:
        for line in f:
            if n >= MAX_RECORDS:
                break
            record = json.loads(line)
            for author in get_authors(record):
                paper_count[author] = paper_count.get(author, 0) + 1
            n += 1
eligible_authors = {author for author, count in paper_count.items() if count >= MIN_PAPERS}
print(f"Total unique authors: {len(paper_count):,}")
print(f"Eligible authors (written >= {MIN_PAPERS} papers): {len(eligible_authors):,}")

Total unique authors: 135,418
Eligible authors (written >= 2 papers): 60,679


## Co-authorship graph

In [11]:
import networkx as nx
import itertools
from collections import defaultdict

edge_weights = defaultdict(int)
n = 0

with zipfile.ZipFile("arxiv.zip", 'r') as z:
    with z.open("arxiv-metadata-oai-snapshot.json") as f:
        for line in f:
            if n >= MAX_RECORDS:
                break
            record = json.loads(line)
            authors = [a for a in get_authors(record) if a in eligible_authors]
            
            for u, v in itertools.combinations(sorted(authors), 2):
              edge_weights[(u, v)] += 1
            
            n += 1

G = nx.Graph()

for (u, v), w in edge_weights.items():
    G.add_edge(u, v, weight=w)
print(f"Graph with {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges.")

Graph with 56,595 nodes and 490,844 edges.
